### Carte combinée : pluie et oxygène dans la Senne

**24 juillet 2025 — 288 mesures, 16 pluviomètres et 4 sondes d’oxygène.**

Objectif : créer une vidéo de **60 secondes** avec la carte animée et les **deux bandes sonores déjà créées** dans les notebooks précédents.

**Important :** cette version mélange deux sonifications existantes. Elle ne crée pas encore une voix sonore individuelle pour chacune des 20 stations. Les cercles et barres représentent, eux, les stations séparément.

Charger et préparer les données

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.animation import FuncAnimation, FFMpegWriter
import imageio_ffmpeg
import matplotlib as mpl
import subprocess

# Chemins relatifs au dossier notebooks/
data = Path('../data')
audio_dir = Path('../audio')
outputs = Path('../outputs')
outputs.mkdir(exist_ok=True)

mesures = pd.read_csv(data / '24_juillet_2025_donnees_preparees.csv')
stations_pluie = pd.read_csv(data / 'coordonnees_pluviometres_flowbru.csv')
stations_oxygene = pd.read_csv(data / 'coordonnees_oxygene_senne.csv')
senne = gpd.read_file(data / 'tracé_senne.geojson').to_crs('EPSG:31370')

mesures.columns = mesures.columns.str.strip()
stations_pluie['colonne_pluie_csv'] = stations_pluie['colonne_pluie_csv'].str.strip()

# Harmoniser automatiquement le nom d'Écluse Anderlecht
colonne_ecluse = next(c for c in mesures.columns if 'anderlecht' in c.lower())
nom_ecluse = stations_pluie.loc[stations_pluie['id_flowbru'] == 'P07', 'colonne_pluie_csv'].iloc[0]
mesures = mesures.rename(columns={colonne_ecluse: nom_ecluse})

colonnes_pluie = stations_pluie['colonne_pluie_csv'].tolist()
colonnes_oxygene = stations_oxygene['capteur'].tolist()
pluie = mesures[colonnes_pluie].apply(pd.to_numeric, errors='coerce')
oxygene = mesures[colonnes_oxygene].apply(pd.to_numeric, errors='coerce')

points_pluie = gpd.GeoDataFrame(stations_pluie, geometry=gpd.points_from_xy(stations_pluie.x_lambert72, stations_pluie.y_lambert72), crs='EPSG:31370')
points_oxygene = gpd.GeoDataFrame(stations_oxygene, geometry=gpd.points_from_xy(stations_oxygene.x_lambert72, stations_oxygene.y_lambert72), crs='EPSG:31370')

assert len(mesures) == 288 and pluie.shape[1] == 16 and oxygene.shape[1] == 4
print('Données prêtes : 288 instants, 16 pluviomètres, 4 sondes d’oxygène.')

Créer la carte animée (60 secondes)

In [ ]:
valeurs_pluie = pluie.to_numpy(dtype=float)
valeurs_oxygene = oxygene.to_numpy(dtype=float)
duree = 60

fig, ax = plt.subplots(figsize=(10, 10))
senne.plot(ax=ax, color='steelblue', linewidth=2)

cercles = ax.scatter(points_pluie.geometry.x, points_pluie.geometry.y,
                     s=np.full(len(points_pluie), 40), color='orange',
                     edgecolor='black', alpha=0.7, label='Pluie')
barres = []
for _, station in points_oxygene.iterrows():
    x, y = station.geometry.x, station.geometry.y
    barre, = ax.plot([x, x], [y, y], color='darkblue', linewidth=7)
    barres.append(barre)
    ax.annotate(station['capteur'], (x, y), xytext=(8, -12),
                textcoords='offset points', fontsize=9)

ax.set_xlim(min(points_pluie.geometry.x.min(), points_oxygene.geometry.x.min()) - 1000,
            max(points_pluie.geometry.x.max(), points_oxygene.geometry.x.max()) + 1000)
ax.set_ylim(min(points_pluie.geometry.y.min(), points_oxygene.geometry.y.min()) - 1000,
            max(points_pluie.geometry.y.max(), points_oxygene.geometry.y.max()) + 2500)
ax.set_aspect('equal')
ax.legend()

def actualiser(i):
    cercles.set_sizes(40 + np.maximum(0, np.nan_to_num(valeurs_pluie[i], nan=0)) * 250)
    for j, barre in enumerate(barres):
        x, y = points_oxygene.geometry.x.iloc[j], points_oxygene.geometry.y.iloc[j]
        valeur = valeurs_oxygene[i, j]
        hauteur = 0 if not np.isfinite(valeur) else max(0, valeur) * 150
        barre.set_data([x, x], [y, y + hauteur])
    ax.set_title(f'Bruxelles — Pluie et oxygène\n{mesures.iloc[i]["Date/heure locale (UTC+2)"]}')
    return [cercles] + barres

animation = FuncAnimation(fig, actualiser, frames=len(mesures),
                          interval=1000 * duree / len(mesures), repeat=False)
plt.close(fig)
print('Animation prête : 288 images sur 60 secondes.')

 Exporter la vidéo sans son

In [ ]:
mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
video_sans_son = outputs / 'pluie_oxygene_sans_son.mp4'
writer = FFMpegWriter(fps=len(mesures)/duree, codec='libx264', bitrate=2500)
animation.save(str(video_sans_son), writer=writer, dpi=120)
print('Vidéo exportée :', video_sans_son)

Mixer les DEUX sons et créer la vidéo finale

Utilisation des fichiers WAV déjà produits : `pluie_24_juillet_2025.wav` et `senne_oxygene.wav`. Le mixage conserve les deux pistes, les ajuste à 60 secondes (avec silence si nécessaire) et évite la saturation par une réduction de niveau.

In [ ]:
son_pluie = audio_dir / 'pluie_24_juillet_2025.wav'
son_oxygene = audio_dir / 'senne_oxygene.wav'
video_finale = outputs / 'pluie_oxygene_DEUX_SONS.mp4'

for fichier in [video_sans_son, son_pluie, son_oxygene]:
    if not fichier.exists():
        raise FileNotFoundError(f'Fichier manquant : {fichier}')

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
commande = [
    ffmpeg, '-y',
    '-i', str(video_sans_son),
    '-i', str(son_pluie),
    '-i', str(son_oxygene),
    '-filter_complex',
    '[1:a]aresample=48000,apad,atrim=duration=60,asetpts=PTS-STARTPTS,volume=0.65[p];'
    '[2:a]aresample=48000,apad,atrim=duration=60,asetpts=PTS-STARTPTS,volume=0.65[o];'
    '[p][o]amix=inputs=2:duration=first:normalize=0,alimiter=limit=0.95[mix]',
    '-map', '0:v:0', '-map', '[mix]',
    '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k',
    '-t', '60', str(video_finale)
]
subprocess.run(commande, check=True, capture_output=True, text=True)
print('VIDÉO FINALE AVEC LES DEUX SONS :', video_finale)

Résultat et limites

La vidéo finale combine **16 pluviomètres** (cercles orange) et **4 sondes d’oxygène** (barres bleues), pour les mesures du **24 juillet 2025**. La durée totale est de **60 secondes**.

La bande sonore résulte du **mixage des deux fichiers WAV des notebooks précédents**. Elle ne permet pas encore d’identifier acoustiquement chaque station. Les choix de représentation restent exploratoires et devront être évalués.
